# Logistic Regression Task

Task 2: Run a Logistic regression in your pairs


Use the titanic sample dataset from seaborn
Predict: Survival rate based on available variables
1. Get & understand the data
 Understand what each passenger variable represents and establish what you're trying to predict.
2. Explore & clean the data
 Check missing values, duplicates, unusual values, categorical variables and class balance.
3. Choose X and y variables
4. Check initial logistic regression assumptions
5. Split into training & test data
6. Create & train the model
7. Make predictions
8. Evaluate model performance
 Confusion matrix
 Accuracy
 Precision
 Recall
 F1 score
 ROC/AUC where appropriate
9. Check model diagnostics & remaining assumptions
10. Interpret, improve & retest the model

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay, classification_report)


### 1. Get and Understand the Data

In [ ]:
df = sns.load_dataset('titanic')

In [ ]:
df.head()

In [ ]:
print(df.shape)

In [ ]:
df.describe()

### 2. Explore and Clean Data

In [ ]:
print("duplicate rows: " , df.duplicated().sum())
print(df.isna().sum())

In [ ]:
df.groupby("sex")["survived"].mean()

In [ ]:
df.groupby("pclass")["survived"].mean()

In [ ]:
df = df.drop_duplicates()

df.duplicated().sum()

In [ ]:
df = df.drop(columns="deck")

### 3. Choose X and Y

In [ ]:
features = ["sex", "age", "pclass", "parch", "fare"]
X = df[features]
Y = df["survived"]

### 4. Check Initial Regression Assumptions

### 5. Split into training & test data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)

In [ ]:
# fill missing ages using the TRAINING median only
age_median = X_train["age"].median()
X_train = X_train.assign(age=X_train["age"].fillna(age_median))
X_test = X_test.assign(age=X_test["age"].fillna(age_median))

# turn text categories into 0/1 columns
X_train = pd.get_dummies(X_train, columns=["sex", "pclass"], drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=["sex", "pclass"], drop_first=True, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.columns.tolist())

### 6. Create & train the model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)

prediction = model.predict(X_test)                 # 0 or 1
probability = model.predict_proba(X_test)[:, 1]    # chance of survival

pd.DataFrame({"actual": Y_test.values, "predicted": prediction,
              "p_survive": probability.round(3)}).head(10)

### 7. Make predictions

In [ ]:
print(confusion_matrix(Y_test, prediction))
print(classification_report(Y_test, prediction))
print("AUC:", roc_auc_score(Y_test, probability))

labels = ["Died", "Survived"]
cm = confusion_matrix(Y_test, prediction)
pct = cm / cm.sum(axis=1, keepdims=True)
annot = np.array([[f"{cm[i, j]}\n({pct[i, j]:.0%})" for j in range(2)] for i in range(2)])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

# confusion matrix
sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False,
            xticklabels=labels, yticklabels=labels,
            linewidths=2, linecolor="white", annot_kws={"size": 14}, ax=ax[0])
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Actual")
ax[0].set_title("Confusion matrix")

# ROC curve
RocCurveDisplay.from_predictions(Y_test, probability, ax=ax[1], name="Logistic regression")
ax[1].plot([0, 1], [0, 1], "k--", label="Random guessing")
ax[1].set_title("ROC curve")
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

### 9.Check model diagnostics & remaining assumptions

### Is the data overfitting?

In [ ]:
print("Train accuracy:", model.score(X_train, Y_train))
print("Test accuracy: ", model.score(X_test, Y_test))


### multicollinearity (are the features too similar?)

In [ ]:
sns.heatmap(X_train.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation between features")
plt.show()

In [ ]:
age_groups = pd.cut(df["age"], [0, 12, 18, 30, 45, 60, 80])
rates = df.groupby(age_groups, observed=True)["survived"].mean()
print(rates.round(2))
rates.plot(kind="bar")
plt.ylabel("Survival rate")
plt.xlabel("Age group")
plt.show()

In [ ]:
check = X_test.copy()
check["actual"] = Y_test
check["p_survive"] = probability.round(2)
check["error"] = (check["actual"] - check["p_survive"]).abs()
check.sort_values("error", ascending=False).head(10)

### 10. improve, retest

In [ ]:
odds = pd.Series(np.exp(model.coef_[0]), index=X_train.columns)
odds.sort_values()

In [ ]:
old = {"accuracy": accuracy_score(Y_test, prediction),
       "precision": precision_score(Y_test, prediction),
       "recall": recall_score(Y_test, prediction),
       "f1": f1_score(Y_test, prediction),
       "auc": roc_auc_score(Y_test, probability)}

In [ ]:
# new feature plus two columns you left out earlier
df["family_size"] = df["sibsp"] + df["parch"] + 1
features2 = ["sex", "age", "pclass", "parch", "fare", "sibsp", "embarked", "family_size"]
X2 = df[features2]

# same split as before, so the comparison is fair
X2_train, X2_test, _, _ = train_test_split(X2, Y, test_size=0.2, random_state=42, stratify=Y)

# fill missing values from the training data, then encode text columns
age_median2 = X2_train["age"].median()
emb_mode = X2_train["embarked"].mode()[0]
X2_train = X2_train.assign(age=X2_train["age"].fillna(age_median2), embarked=X2_train["embarked"].fillna(emb_mode))
X2_test = X2_test.assign(age=X2_test["age"].fillna(age_median2), embarked=X2_test["embarked"].fillna(emb_mode))
X2_train = pd.get_dummies(X2_train, columns=["sex", "pclass", "embarked"], drop_first=True, dtype=float)
X2_test = pd.get_dummies(X2_test, columns=["sex", "pclass", "embarked"], drop_first=True, dtype=float)
X2_test = X2_test.reindex(columns=X2_train.columns, fill_value=0)

# train and predict
model2 = LogisticRegression(max_iter=1000)
model2.fit(X2_train, Y_train)
prediction2 = model2.predict(X2_test)
probability2 = model2.predict_proba(X2_test)[:, 1]

new = {"accuracy": accuracy_score(Y_test, prediction2),
       "precision": precision_score(Y_test, prediction2),
       "recall": recall_score(Y_test, prediction2),
       "f1": f1_score(Y_test, prediction2),
       "auc": roc_auc_score(Y_test, probability2)}

pd.DataFrame({"before": old, "after": new}).round(3)